In [ ]:
import sys
sys.path.append('..')
sys.path.append("../College/DataPrep/")
sys.path.append("../College/Model/")

import importlib
from College.DataPrep import Prep_Map, Output_Map, Player_Dataset, Data_Prep
importlib.reload(Data_Prep)
importlib.reload(Prep_Map)
importlib.reload(Output_Map)
importlib.reload(Player_Dataset)
from College.DataPrep.Data_Prep import College_Data_Prep, College_IO
from College.DataPrep.Player_Dataset import College_Player_Dataset, Create_Test_Train_Datasets
from Constants import device

In [ ]:
batch_size = 800
num_epochs = 100

In [ ]:
import torch
torch.set_printoptions(precision=8, sci_mode=False)
torch.set_printoptions(
    precision=2,
    sci_mode=False,
    linewidth=500,
    edgeitems=20,
)

In [ ]:
num_layers = 2
hidden_size = 100

In [ ]:
data_prep = College_Data_Prep(Prep_Map.college_base_prep_map, Output_Map.college_output_map)
hitter_io_list = data_prep.Generate_IO_Hitters("WHERE LastYear<=? AND isHitter=?", (2019, 1), use_cutoff=True)
train_dataset, test_dataset = Create_Test_Train_Datasets(hitter_io_list, 0.25, 0)

In [ ]:
from College.Model import College_Model, Model_Train
importlib.reload(College_Model)
importlib.reload(Model_Train)
from torch.optim import lr_scheduler
from College.Model.College_Model import RNN_Model
from College.Model.Model_Train import trainAndGraph

network = RNN_Model(train_dataset.get_input_size(), 
                    num_layers, 
                    hidden_size, 
                    data_prep=data_prep, 
                    is_hitter=True)
network = network.to(device)

optimizer = torch.optim.Adam(network.parameters(), lr=0.0025)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5, cooldown=5, verbose=False)

best_losses = trainAndGraph(network,
              train_dataset,
              test_dataset,
              scheduler,
              optimizer,
              batch_size,
              num_epochs = num_epochs,
              logging_interval=10,
              early_stopping_cutoff=2000,
              should_output=True,
              model_name="../Models/default_college_hitter",
              save_last=True,
              elements_to_save=[0])

print(best_losses)